In [23]:
print("hola")

hola


In [24]:
import pyodbc

def obtener_conexion():
    server = "172.16.1.33"
    database = "CUN_REPOSITORIO"
    user = "coe"
    password = "C6kx9nPwTkuH-y6WW.BT"

    conn_str = (
        "DRIVER={ODBC Driver 17 for SQL Server};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"UID={user};"
        f"PWD={password};"
        "Trusted_Connection=no;"
    )
    
    try:
        conn = pyodbc.connect(conn_str)
        # ESTO SOLO SE IMPRIME SI LA FUNCIÓN SE EJECUTA
        print("✅ Conexión exitosa con el usuario COE.")
        return conn
    except Exception as e:
        print(f"❌ ERROR AL CONECTAR: {e}")
        return None


mi_conexion = obtener_conexion()

✅ Conexión exitosa con el usuario COE.


In [26]:
import pandas as pd
from tabulate import tabulate 

conn = obtener_conexion()

if conn:
    query = """
        SELECT * FROM ZOHO.dbo.VW_Tickets
        WHERE RegionalSede = 'Neiva' 
        AND CreatedTime >= '2025-01-01'
        ORDER BY CreatedTime ASC
    """
    #C:\Users\juan_garnicac\Documents\ProyectosVisual\EschuahEnvivo\febrero-12\consultas\conexion.ipynb
    
    df = pd.read_sql(query, conn)

    if not df.empty:
        # Aquí quitamos el [cols] para que use TODO el dataframe
        print(tabulate(df.head(50), headers='keys', tablefmt='psql', showindex=False))
        
        nombre_archivo = "Tickets_Neiva_TOTAL.xlsx"
        # Aquí también quitamos el [cols] para que el Excel guarde todo
        df.to_excel(nombre_archivo, index=False)
        print(f"\n✅ Excel guardado con TODAS las columnas: {nombre_archivo}")
    else:
        print("⚠ No se encontraron tickets para Sincelejo en ese rango.")

    conn.close()

print("\n📌 Lista completa de columnas capturadas:")
print(df.columns)

✅ Conexión exitosa con el usuario COE.


C:\Users\juan_garnicac\AppData\Local\Temp\ipykernel_30504\442492624.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


+--------------------+----------------+---------------------+---------------------+---------------------+---------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------+------------------------------------------------+----------------------------------------------------------------------------------+----------------------+-------------+----------------------------------+---------------+----------------+---------------------------------+-------------------+-------------------------------+----------------------------+----------------+-----------+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [22]:
import pandas as pd
import re

def clasificar_ticket(texto):
    """
    Clasifica un ticket según las categorías de graficos.js
    """
    t = str(texto).lower()
    
    # ==========================================
    # ACADÉMICO - Calidad Pensum
    # ==========================================
    if re.search(r'\bpensum\b|\bmalla curricular\b|\bplan de estudios\b|\bactualizacion maya\b', t):
        return ("Calidad del pensul")
    
    if re.search(r'\bactualizacion\b|\bactualizar\b|\bmateria obsoleta\b|\bcontenido no vigente\b', t):
        return ("Actualizacion del pensum")
    
    # ==========================================
    # ACADÉMICO - Docencia
    # ==========================================
    if re.search(r'\bdocente\b|\bprofesor\b|\bpedagogia\b|\bexplicacion\b|\basistencia\b|\bcambio de docente\b|\bno entiendo\b', t):
        return ("Docencia - Poca explicación y asistencia")
    
    if re.search(r'\bclase pesima\b|\bmala clase\b|\bclase terrible\b|\bclase aburrida\b', t):
        return ("Clases pésimas")
    
    # ==========================================
    # ACADÉMICO - Prácticas Profesionales (8 subcategorías)
    # ==========================================
    if re.search(r'\bnota practica\b|\bcalificacion practica\b|\bevaluacion practica\b', t):
        return ("Calificación y nota de prácticas")
    
    if re.search(r'\berror plataforma\b|\bno aparece practica\b|\bno se registra practica\b|\bplataforma practicas falla\b', t):
        return ("Error en plataforma / No aparece o no se registra")
    
    if re.search(r'\bmatricula practica\b|\boferta practica\b|\binscripcion practica\b|\bcupo practica\b', t):
        return ("Matrícula / Oferta de prácticas")
    
    if re.search(r'\basignacion docente\b|\bhorario practica\b|\bdocente practica\b', t):
        return ("Asignación de docente / Horario")
    
    if re.search(r'\bsitio practicas\b|\bconvenio\b|\bproblemas sitio\b|\bdemora convenio\b|\barl\b', t):
        return ("Problemas con sitio de prácticas / Convenios")
    
    if re.search(r'\bajuste fechas\b|\bfecha practica\b|\bregistro practica\b', t):
        return ("Ajuste de fechas y registro")
    
    if re.search(r'\busuario sgp\b|\bcontraseña sgp\b', t):
        return ("Usuario/contraseña incorrecto en SGP")
    
    if re.search(r'\bconvenio empresa\b', t):
        return ("Convenio con empresa")
    
    # ==========================================
    # ACADÉMICO - Otras
    # ==========================================
    if re.search(r'\bgrado\b|\btitulo\b|\bdiploma\b|\bgraduacion\b', t):
        return ("GRADOS Y TITULACIONES")
    
    if re.search(r'\bnota\b|\bcalificacion\b|\bquiz\b|\bparcial\b|\baca\b', t):
        return ("PRACTICAS")
    
    # ==========================================
    # TECNOLÓGICO
    # ==========================================
    if re.search(r'\bcontraseña\b|\bclave\b|\bacceso\b|\bdesbloquear\b|\breestablecer\b|\brestablecer\b', t):
        return ("Contraseña / Clave / Acceso")
    
    if re.search(r'\bcorreo institucional\b|\bemail\b', t):
        return ("Correo Institucional")
    
    if re.search(r'\bcdigital\b|\bcun digital\b|\bbiblioteca virtual\b|\bplataforma cdigital\b', t):
        return ("C-Digital / Biblioteca Virtual")
    
    if re.search(r'\bsinu\b', t):
        return ("SINU / Usuario y contraseña")
    
    if re.search(r'\bplataforma\b|\bno puedo entrar\b|\bingreso\b|\bportal\b', t):
        return ("Acceso general a plataformas")
    
    if re.search(r'\btyt\b|\bpruebas tyt\b|\bsaber tyt\b|\bbloqueo\b.*\bdocumentos\b', t):
        return ("Bloqueo por documentos / TYT")
    
    # ==========================================
    # BIENESTAR
    # ==========================================
    if re.search(r'\bbeca\b|\bsubsidio\b|\bauxilio\b|\bapoyo economico\b', t):
        return ("Becas deportivas y culturales")
    
    if re.search(r'\bbienestar\b|\bpermanencia\b|\bconectividad\b', t):
        return ("Beneficios ofrecidos por Bienestar")
    
    if re.search(r'\bdificultad\b|\bsemestre\b|\bproblemas\b|\batrasado\b', t):
        return ("Dificultades semestre en curso")
    
    if re.search(r'\bacompañamiento\b|\bapoyo familiar\b|\btutoria\b', t):
        return ("Acompañamientos académicos / Apoyo familiar")
    
    if re.search(r'\bactualizacion\b.*\bdatos\b|\bsaldo\b.*\bremision\b', t):
        return ("Otros (actualización datos, remisión saldo)")
    
    # ==========================================
    # FINANCIERO (para CamiTicket)
    # ==========================================
    if re.search(r'\bpago\b|\bmatricula\b|\brecibo\b|\bdeuda\b|\bsaldo\b|\bfinanciero\b', t):
        return ("FINANCIERO")
    
    # ==========================================
    # PQRS
    # ==========================================
    if re.search(r'\bticket\b|\bcami\b|\bpqrs\b|\bpeticion\b|\bqueja\b', t):
        return ("PQRS")
    
    # ==========================================
    # SERVICIOS
    # ==========================================
    if re.search(r'\bclimatizacion\b|\baire acondicionado\b', t):
        return ("Climatización")
    
    if re.search(r'\bcafeteria\b|\bcomedor\b', t):
        return ("Cafetería")
    
    if re.search(r'\benfermeria\b', t):
        return ("Enfermería")
    
    # ==========================================
    # DEFAULT
    # ==========================================
    # Si no clasificó, usar la parte después del guion o el texto completo
    if ' - ' in texto:
        parte = texto.split(' - ')[-1].strip()
        if len(parte) > 3:
            return (parte)
    
    return ("Sin clasificar")

# ==================================================
# CLASIFICAR Y GUARDAR
# ==================================================

# Asumiendo que su DataFrame se llama 'df' y la columna de texto se llama 'Subject1'
df_clasificacion = df["Subject1"]

resultados = []
for texto in df_clasificacion:
    subcategoria = clasificar_ticket(texto)
    resultados.append({
        "texto": texto,
        "subcategoria": subcategoria
    })

df_final = pd.DataFrame(resultados)

# Guardar a Excel
df_final.to_excel("tickets_clasificados.xlsx", index=False)

print("✅ Archivo guardado: tickets_clasificados.xlsx")
print("\n=== Conteo por subcategoría ===")
print(df_final['subcategoria'].value_counts())

✅ Archivo guardado: tickets_clasificados.xlsx

=== Conteo por subcategoría ===
subcategoria
FINANCIERO                                     449
GRADOS Y TITULACIONES                          198
CERTIFICADOS                                   114
Dificultades semestre en curso                 102
Contraseña / Clave / Acceso                     97
                                              ... 
ERROR INFORMACIÓN OBSERVATORIO LABORAL           1
Acompañamientos académicos / Apoyo familiar      1
PAGAR CUOTA DE CT AYUDA                          1
INCONSISTENCIAS EN NOTAS Y/O CALIFICACIONES      1
INSUMOS                                          1
Name: count, Length: 147, dtype: int64
